# Bounds-theorem grid analysis

**Purpose.** This notebook evaluates theorem-backed quantitative bounds on
Sobol index shifts under smooth output transformations, following the
theoretical framework of:

> Hettinger, D. (2026). *Quantitative Bounds on Sobol Index Shifts Under
> Smooth Output Transformations.* (Companion memo to the noncommutativity
> paper.)  Source: `docs/manuscripts/bounds_memo_v22.tex`.

The companion commutativity analysis (notebook
`noncommutativity_grid_analysis.ipynb`) asks *whether* a transform changes
Sobol indices.  This notebook asks *how much* it can change them, using a
mathematically rigorous perturbation bound.

**Critical interpretation note.** Bounds in this notebook compare the Sobol
indices of the transformed output $Z = \phi(Y)$ to the Sobol indices of the
**Taylor reference** $V_m$ — not directly to those of $Y$.  This is the
correct object of comparison from the theorem's perspective; see the
*Mathematical framework* section below for why.

**What this notebook produces.**
Running all cells creates four CSV files in
`outputs/bounds_theorem_grid_analysis/` (or the directory set by
`SABENCH_BOUNDS_OUTPUT_DIR`):

| File | Contents |
|------|----------|
| `bounds_pair_status.csv` | One row per (benchmark, transform) pair; status and metadata |
| `taylor_reference_results.csv` | Taylor-reference diagnostics for eligible pairs |
| `local_affine_results.csv` | Local-affine ($m=1$) diagnostics for eligible pairs |
| `bounds_summary.csv` | Count of each status across the full grid |

**How to run.**
```bash
# Fast smoke run (default — CI-safe, small N)
jupyter notebook notebooks/bounds_theorem_grid_analysis.ipynb

# Full run (all compatible pairs, larger sample)
SABENCH_BOUNDS_N_BASE=1024 jupyter notebook notebooks/bounds_theorem_grid_analysis.ipynb

# Limit scope for testing
SABENCH_BOUNDS_MAX_BENCHMARKS=5 SABENCH_BOUNDS_MAX_TRANSFORMS=20 \
  jupyter notebook notebooks/bounds_theorem_grid_analysis.ipynb
```

**Dependencies.** All scientific logic is in tested `sabench.analysis.bounds`
and `sabench.analysis.bounds_grid` modules.  No notebook-local scientific
computations.

## Background: the quantitative perturbation question

The companion noncommutativity analysis (Proposition 1 of the paper) proves a
**qualitative** statement: only affine transforms commute with Sobol index
computation for every admissible model.  Every nonlinear transform can change
the sensitivity profile.

This notebook addresses the complementary **quantitative** question:

> For a specific smooth transform $\phi$ applied to a specific benchmark, how
> large a Sobol index shift is possible, given the output distribution of that
> benchmark?

The answer uses a perturbation bound.  Instead of comparing $\phi(Y)$ directly
to $Y$, the theorem compares $\phi(Y)$ to the best polynomial approximation of
$\phi(Y)$ in the vicinity of the output mean — the **Taylor reference** $V_m$.
This choice is natural because:

1. When $\phi$ is exactly a degree-$m$ polynomial, the Taylor residual is zero
   and the bound is exact.
2. When $\phi'(\mu_Y) \neq 0$ (nonzero slope at the mean), the local-affine
   reference $V_1 = \phi'(\mu_Y)(Y - \mu_Y)$ has the same Sobol structure as
   $Y$, so bounding the distance from $\phi(Y)$ to $V_1$ gives an indirect
   bound on the distance from $\phi(Y)$ to $Y$.
3. When $\phi'(\mu_Y) = 0$ (a critical point), comparing $\phi(Y)$ directly
   to $Y$ may be misleading because $Y$ and $\phi(Y)$ may have structurally
   different sensitivity profiles regardless of how smooth $\phi$ is.  The
   Taylor reference with $m=2$ gives the appropriate comparator.

**Scope of this notebook.** Only smooth (i.e., several-times differentiable)
pointwise transforms have registered derivative metadata in the `sabench`
registry; the bounds computation is skipped for nonsmooth, non-pointwise, or
multi-output-benchmark pairs.  The grid covers all registered smooth+pointwise
transforms applied to all registered benchmarks.

## Mathematical framework: Taylor reference and residual

### Setup

Let $Y = f(\mathbf{X})$ be a scalar model output and $\phi : \mathbb{R} \to
\mathbb{R}$ a smooth transform.  Write $Z = \phi(Y)$ and $\mu_Y = \mathbb{E}[Y]$.

### Taylor reference $V_m$

The order-$m$ Taylor reference is the $m$-th degree Taylor polynomial of
$\phi$ expanded around $\mu_Y$, applied to the centered output:

$$V_m = \sum_{k=1}^{m} \frac{\phi^{(k)}(\mu_Y)}{k!} (Y - \mu_Y)^k.$$

Note: $V_m$ does *not* include the constant term $\phi(\mu_Y)$; it is the
centered, polynomial-approximated part of $\phi(Y)$.

### Taylor residual $R_m$

$$R_m = \phi(Y) - \phi(\mu_Y) - V_m.$$

$R_m$ is the error of the Taylor approximation, shifted to have the same mean
as $V_m$ (both are zero-mean to the extent $\phi$ is well-approximated).

### Empirical $\eta_m$

The key dimensionless quantity is the ratio of the residual's standard
deviation to the reference's standard deviation:

$$\eta_m = \frac{\mathrm{sd}(R_m - \mathbb{E}[R_m])}{\mathrm{sd}(V_m)}.$$

- $\eta_m = 0$: $\phi$ is exactly a degree-$m$ polynomial; the bound is
  exact with zero error.
- $\eta_m < 1$: the bound is finite and meaningful.
- $\eta_m \geq 1$: the residual dominates the reference; the abstract bound
  degrades to $> 1$ and provides no useful information.

### Sufficient $\eta$ upper bound $\bar{\eta}_m$

When the transform has a known derivative supremum
$\rho = \sup_{y \in [y_-, y_+]} |\phi^{(m+1)}(y)|$ over the support $[y_-, y_+]$,
Taylor's theorem gives a computable upper bound:

$$\bar{\eta}_m \leq
  \frac{\rho \cdot \sqrt{\mathbb{E}[|Y - \mu_Y|^{2m+2}]}}{(m+1)! \cdot \mathrm{sd}(V_m)}.$$

If $\bar{\eta}_m < 1$, the abstract bound is valid with $\bar{\eta}_m$ in
place of $\eta_m$.  This requires a **provably correct support interval**
$[y_-, y_+]$ that contains the output range almost surely.  The notebook
provides this through `BENCHMARK_OUTPUT_BOUNDS`.

### Abstract projection bound

Given $\eta_m < 1$ and a Sobol index subset $\mathscr{C}$, the projection
error satisfies:

$$\bigl|\mathsf{PE}_{\mathscr{C}}(Z) - \mathsf{PE}_{\mathscr{C}}(V_m)\bigr|
  \leq
  \frac{2\eta_m \sqrt{p}\,(1 + \sqrt{p}) + \eta_m^2(1 + p)}{(1 - \eta_m)^2},$$

where $p = \mathsf{PE}_{\mathscr{C}}(V_m)$ is the Taylor reference's projection
value and $\mathsf{PE}$ is the normalized-projection shorthand for ANOVA
component, closed, or total-effect Sobol indices.

*In the notebook outputs, the bound is computed for singleton and
total-effect projection classes and reported as `projection_bound_s1_*` and
`projection_bound_st_*` columns.*

## Local-affine diagnostics ($m = 1$ case)

When $\phi'(\mu_Y) \neq 0$ — the **nonzero-slope** case — the $m=1$ Taylor
reference is:

$$V_1 = \phi'(\mu_Y)\,(Y - \mu_Y).$$

$V_1$ is an affine function of $Y$, so $V_1$ has the *same* Sobol structure as
$Y$ itself (all indices are identical to those of $Y$).  This means the
abstract projection bound in the $m=1$ case also bounds the distance from
$\phi(Y)$'s Sobol indices to those of $Y$.

### The diagnostic quantities $K$, $\kappa$, $\lambda$

The local-affine bound uses three benchmark- and transform-specific quantities:

**Moment ratio** $K$ (characterises the output distribution's heavy-tailedness):

$$K = \frac{\sqrt{\mu_4}}{\sigma_Y^2},$$

where $\mu_4 = \mathbb{E}[(Y - \mu_Y)^4]$ is the fourth central moment and
$\sigma_Y^2 = \mathrm{Var}[Y]$.  Note: $K$ is estimated empirically from the
Saltelli sample.

**Nonlinearity ratio** $\kappa$ (characterises how curved $\phi$ is relative
to the output's scale):

$$\kappa = \frac{\rho_2\,\sigma_Y}{|\phi'(\mu_Y)|},$$

where $\rho_2 = \sup_{y \in [y_-, y_+]} |\phi''(y)|$ is the second-derivative
supremum over the output support.

**Composite diagnostic** $\lambda = K \kappa$:

$$\lambda = K \kappa = \frac{\sqrt{\mu_4}}{\sigma_Y^2}
  \cdot \frac{\rho_2\,\sigma_Y}{|\phi'(\mu_Y)|}
  = \frac{\rho_2\sqrt{\mu_4}}{\sigma_Y\,|\phi'(\mu_Y)|}.$$

The local-affine Corollary states that $\eta_1 \leq \lambda/2$.  If
$\lambda < 2$ (equivalently $\eta_1 < 1$), the local-affine bound is finite:

$$\bigl|\mathsf{PE}_{\mathscr{C}}(Z) - \mathsf{PE}_{\mathscr{C}}(Y)\bigr|
  \leq
  \frac{\lambda\sqrt{p}\,(1 + \sqrt{p}) + \tfrac14\lambda^2(1 + p)}{(1 - \lambda/2)^2}.$$

**Interpretation of $\lambda$:**
- $\lambda \ll 1$: the transform is nearly affine over the output range;
  Sobol shifts are expected to be small.
- $\lambda \approx 1$: moderate nonlinearity; the bound is finite but loose.
- $\lambda \geq 2$: the sufficient stability condition fails; the local-affine
  bound may still hold with the empirical $\eta_1$, but the analytic guarantee
  via $\bar{\eta}$ is unavailable.

### When the slope is zero (`zero_slope`)

If $\phi'(\mu_Y) = 0$ (a critical point), the local-affine reference $V_1 = 0$
is degenerate.  The $m=1$ bound is not meaningful; use $m=2$ (quadratic
reference) instead.  The `local_affine_status = "zero_slope"` flag marks this
case.

## Status vocabulary

Every row in `bounds_pair_status.csv` carries a `bounds_status` value that
records *why* the row reached the outcome it did.  Understanding these statuses
is important for correctly interpreting the coverage of the grid.

| Status | Meaning |
|--------|---------|
| `bounds_supported` | Caller-provided support bounds were used (from `BENCHMARK_OUTPUT_BOUNDS`).  Taylor-reference diagnostics were computed with a rigorous support interval.  **This is the strongest status.**  Note: the support interval itself may be analytically derived (e.g., Ishigami, Friedman) or empirically conservative (e.g., Borehole: lo=0 analytic, hi from N=1M + 5% buffer). |
| `bounds_diagnostic_sample_support` | Diagnostics were computed, but the support interval used was the empirical sample range (min/max of the Saltelli sample).  These are valid sample-range diagnostics but **not** theorem guarantees, because sample extremes may not contain the true support. |
| `bounds_not_scalar_output` | Benchmark produces multi-output (spatial/temporal) results; the bounds framework requires a scalar output. |
| `bounds_not_pointwise` | Transform is not pointwise (e.g., block average, COS); the bounds framework requires a pointwise function $\phi$. |
| `bounds_not_smooth` | Transform is classified as nonsmooth (e.g., threshold, rectifier, absolute value); derivative metadata is not registered. |
| `bounds_no_derivative_metadata` | Transform is smooth and pointwise but lacks registered derivative information.  As of PR #11, this count is 0 for all catalog smooth+pointwise transforms. |
| `bounds_domain_invalid` | Benchmark output falls outside the transform's valid domain (e.g., log of a negative output). |
| `bounds_reference_zero_variance` | The Taylor reference $V_m$ has zero variance.  This occurs when all Taylor terms cancel, making the bound degenerate; increase the Taylor order. |
| `bounds_eta_ge_one` | Empirical $\eta_m \geq 1$; the residual dominates the reference and the bound provides no finite information. |
| `bounds_failed` | An unexpected runtime error occurred; inspect the `reason` column. |

### How `bounds_supported` works in practice

This notebook passes `benchmark_support=BENCHMARK_OUTPUT_BOUNDS` to
`evaluate_bounds_grid()`.  `BENCHMARK_OUTPUT_BOUNDS` contains entries for all
19 registered scalar benchmarks, so every smooth+pointwise pair with a scalar
benchmark will reach `bounds_supported` status.  Pairs with multi-output
benchmarks will reach `bounds_not_scalar_output`, and pairs with nonsmooth
or non-pointwise transforms will reach their respective statuses.

**Support provenance for `BENCHMARK_OUTPUT_BOUNDS`:**
- *Analytically exact* (12 benchmarks): Ishigami, SobolG, LinearModel,
  AdditiveQuadratic, CornerPeak, Friedman, MoonHerrera, DetPep8D, Rosenbrock,
  ProductPeak, PCETestFunction, CSTRReactor.
- *Empirically conservative, lower bound analytic* (5 benchmarks, all
  provably non-negative): Borehole, Piston, WingWeight, OTLCircuit,
  EnvironModel.  Upper bound from N=1M sample + 5% buffer.
- *Empirically conservative, both sides* (2 benchmarks): Morris, OakleyOHagan.
  Both bounds from N=1M sample + 5% buffer.

For rigorous theorem applications, use the analytically exact entries.
For the empirically conservative entries, treat results as strong diagnostics
but acknowledge the support interval is not provably tight.

## Configuration

| Parameter | Env var | Default | Meaning |
|-----------|---------|---------|---------|
| `N_BASE` | `SABENCH_BOUNDS_N_BASE` | 128 | Base sample size for Saltelli design; actual evaluations = $N(d+2)$.  Use 512–2048 for publication-quality diagnostics. |
| `RNG_SEED` | `SABENCH_BOUNDS_SEED` | 20260429 | Random seed.  Fixed across all pairs. |
| `TAYLOR_ORDER` | `SABENCH_BOUNDS_TAYLOR_ORDER` | 2 | Order $m$ of the Taylor reference.  Order 1 = local-affine reference only.  Order 2 includes the quadratic term, which matters near critical points ($\phi'(\mu_Y) = 0$). |
| `MAX_BENCHMARKS` | `SABENCH_BOUNDS_MAX_BENCHMARKS` | 0 (all) | Truncate benchmark list to first $n$ entries.  0 = use all. |
| `MAX_TRANSFORMS` | `SABENCH_BOUNDS_MAX_TRANSFORMS` | 0 (all) | Truncate transform list to first $n$ entries.  0 = use all. |
| `OUTPUT_DIR` | `SABENCH_BOUNDS_OUTPUT_DIR` | `outputs/bounds_theorem_grid_analysis` | Directory for exported CSVs.  Created if absent. |

**Taylor order choice.**  Order 2 is appropriate for most purposes: it covers
the local-affine case ($m=1$ subcalculation) and the quadratic correction.
For exactly polynomial transforms of known degree, set the order to that
degree to get an exact (residual = 0) result.

**Transform ordering.** The configuration cell places smooth+pointwise
transforms first so that `MAX_TRANSFORMS` retains the most theoretically
interesting subset.

**Note on `bounds_supported` scope.** Only smooth+pointwise transforms with
scalar benchmarks produce `bounds_supported` rows.  The total grid includes
all registered transforms (smooth+pointwise ∪ nonsmooth ∪ non-pointwise);
non-applicable pairs are retained as status rows for completeness.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from sabench.analysis.bounds import supported_smooth_pointwise_transform_keys
from sabench.analysis.bounds_grid import BENCHMARK_OUTPUT_BOUNDS, BOUNDS_STATUSES, evaluate_bounds_grid
from sabench.benchmarks import BENCHMARK_REGISTRY
from sabench.transforms import TRANSFORM_REGISTRY

N_BASE = int(os.environ.get("SABENCH_BOUNDS_N_BASE", "128"))
RNG_SEED = int(os.environ.get("SABENCH_BOUNDS_SEED", "20260429"))
TAYLOR_ORDER = int(os.environ.get("SABENCH_BOUNDS_TAYLOR_ORDER", "2"))
MAX_BENCHMARKS = int(os.environ.get("SABENCH_BOUNDS_MAX_BENCHMARKS", "0"))
MAX_TRANSFORMS = int(os.environ.get("SABENCH_BOUNDS_MAX_TRANSFORMS", "0"))

_HERE = Path.cwd()
_REPO_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE
OUTPUT_DIR = Path(
    os.environ.get(
        "SABENCH_BOUNDS_OUTPUT_DIR",
        str(_REPO_ROOT / "outputs" / "bounds_theorem_grid_analysis"),
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_keys = tuple(BENCHMARK_REGISTRY)
supported_transform_keys = tuple(
    key for key in supported_smooth_pointwise_transform_keys() if key in TRANSFORM_REGISTRY
)
remaining_transform_keys = tuple(
    key for key in TRANSFORM_REGISTRY if key not in set(supported_transform_keys)
)
transform_keys = supported_transform_keys + remaining_transform_keys

if MAX_BENCHMARKS > 0:
    benchmark_keys = benchmark_keys[:MAX_BENCHMARKS]
if MAX_TRANSFORMS > 0:
    transform_keys = transform_keys[:MAX_TRANSFORMS]

print(
    f"Bounds grid: {len(benchmark_keys)} benchmarks × {len(transform_keys)} transforms, "
    f"N_BASE={N_BASE}, Taylor order={TAYLOR_ORDER}."
)

## Execute the theorem-assumption grid

The `evaluate_bounds_grid()` function processes each (benchmark, transform)
pair through the following pipeline:

1. **Applicability check** — verifies scalar output, pointwise mechanism,
   smooth tags, and derivative metadata.  Non-applicable pairs receive a
   descriptive status and no computation is attempted.

2. **Support resolution** — determines the support interval $[y_-, y_+]$:
   - If the benchmark key is in `BENCHMARK_OUTPUT_BOUNDS` (passed via
     `benchmark_support`): use the provided interval → `support_source = "provided_support"`.
   - Otherwise: use the sample range (min/max of the Saltelli output) →
     `support_source = "empirical_sample_range"`.

3. **Taylor reference computation** — evaluates $V_m$, $R_m$, and $\eta_m$
   empirically; computes the sufficient $\bar{\eta}_m$ if derivative
   supremum is available.

4. **Projection bound computation** — evaluates the abstract bound formula
   for first-order and total-effect Sobol index subsets.

5. **Local-affine diagnostics** — computes $K$, $\kappa$, $\lambda$, and
   $\eta_1$ for the $m=1$ case; reports `zero_slope` if $\phi'(\mu_Y) = 0$.

Pair-level failures (domain errors, numeric issues) are caught and recorded
as structured status rows rather than raising exceptions.

After the cell runs, `df_bounds` contains one row per pair.  The
`bounds_status` column records the outcome; numeric diagnostic columns are
populated only for applicable rows.

In [ ]:
results = evaluate_bounds_grid(
    benchmark_keys=benchmark_keys,
    transform_keys=transform_keys,
    n_base=N_BASE,
    seed=RNG_SEED,
    taylor_order=TAYLOR_ORDER,
    benchmark_support=BENCHMARK_OUTPUT_BOUNDS,
)
rows = [result.as_dict() for result in results]
df_bounds = pd.DataFrame(rows)

print(f"Evaluated {len(df_bounds)} candidate pairs")
print(df_bounds["bounds_status"].value_counts().to_string())

## Exported tables and column glossary

### `bounds_pair_status.csv`

One row per (benchmark, transform) pair, regardless of applicability.

| Column | Description |
|--------|-------------|
| `benchmark_key` | Registry key |
| `transform_key` | Registry key |
| `bounds_status` | Primary status (see *Status vocabulary* above) |
| `reason` | Human-readable reason for non-`bounds_supported` rows |
| `benchmark_output_kind` | `scalar`, `spatial`, or `temporal` |
| `transform_mechanism` | `pointwise`, `spatial`, etc. |
| `transform_tags` | Semicolon-separated tags |
| `n_base`, `seed`, `n_inputs`, `n_evaluations` | Run parameters |
| `output_shape`, `output_finite`, `output_variance` | Output diagnostics |
| `taylor_order` | Taylor polynomial order used |

### `taylor_reference_results.csv`

Rows for all pairs that reached the Taylor-reference computation step
(i.e., `support_source` is not null).

| Column | Description |
|--------|-------------|
| `bounds_status` | `bounds_supported` or `bounds_diagnostic_sample_support` |
| `support_source` | `"provided_support"` (analytical/conservative bounds) or `"empirical_sample_range"` |
| `support_lower`, `support_upper` | The support interval $[y_-, y_+]$ used |
| `taylor_status` | `"computed"`, `"zero_reference_variance"`, or `"failed"` |
| `eta_empirical` | Empirical $\eta_m = \mathrm{sd}(R_m)/\mathrm{sd}(V_m)$ |
| `eta_sufficient` | Sufficient upper bound $\bar{\eta}_m$ (or null if derivative supremum unavailable) |
| `eta_empirical_lt_one` | Boolean: whether $\eta_m < 1$ (bound is finite) |
| `eta_sufficient_lt_one` | Boolean: whether $\bar{\eta}_m < 1$ |
| `projection_bound_s1_max` | Maximum first-order projection bound across inputs |
| `projection_bound_st_max` | Maximum total-effect projection bound across inputs |
| `projection_bound_s1_mean` | Mean first-order projection bound |
| `projection_bound_st_mean` | Mean total-effect projection bound |
| `reference_shift_s1_max` | Maximum first-order Sobol shift from $V_m$ to $Z$ |
| `reference_shift_st_max` | Maximum total-effect Sobol shift from $V_m$ to $Z$ |
| `reference_shift_s1_mean` | Mean first-order shift |
| `reference_shift_st_mean` | Mean total-effect shift |

### `local_affine_results.csv`

Rows for pairs that reached the local-affine diagnostic step.

| Column | Description |
|--------|-------------|
| `local_affine_status` | `"computed"` or `"zero_slope"` |
| `support_source` | Same as above |
| `sigma_y` | Standard deviation of benchmark output, $\sigma_Y$ |
| `mu4` | Fourth central moment $\mu_4 = \mathbb{E}[(Y-\mu_Y)^4]$ |
| `phi_prime_mu` | First derivative $\phi'(\mu_Y)$ |
| `rho2` | Second-derivative supremum $\rho_2 = \sup|\phi''|$ over $[y_-, y_+]$ |
| `kappa` | Nonlinearity ratio $\kappa = \rho_2\sigma_Y / |\phi'(\mu_Y)|$ |
| `moment_ratio` | Moment ratio $K = \sqrt{\mu_4}/\sigma_Y^2$ |
| `lambda_value` | Composite diagnostic $\lambda = K\kappa$ |
| `eta_upper` | $\eta_1 \leq \lambda/2$ (upper bound on first-order eta) |
| `lambda_lt_two` | Boolean: whether $\lambda < 2$ (finite local-affine bound exists) |

### `bounds_summary.csv`

Two-column table (`bounds_status`, `count`) reporting how many rows of each
status appeared.  Ordered by the canonical status list for reproducibility.

In [ ]:
from collections import Counter

PAIR_STATUS_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "bounds_status",
    "reason",
    "benchmark_output_kind",
    "transform_mechanism",
    "transform_tags",
    "n_base",
    "seed",
    "n_inputs",
    "n_evaluations",
    "output_shape",
    "output_finite",
    "output_variance",
    "taylor_order",
]
TAYLOR_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "bounds_status",
    "support_source",
    "support_lower",
    "support_upper",
    "taylor_status",
    "eta_empirical",
    "eta_sufficient",
    "eta_empirical_lt_one",
    "eta_sufficient_lt_one",
    "projection_bound_s1_max",
    "projection_bound_st_max",
    "projection_bound_s1_mean",
    "projection_bound_st_mean",
    "reference_shift_s1_max",
    "reference_shift_st_max",
    "reference_shift_s1_mean",
    "reference_shift_st_mean",
]
LOCAL_AFFINE_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "bounds_status",
    "local_affine_status",
    "support_source",
    "sigma_y",
    "mu4",
    "phi_prime_mu",
    "rho2",
    "kappa",
    "moment_ratio",
    "lambda_value",
    "eta_upper",
    "lambda_lt_two",
]

status_counts = Counter(df_bounds["bounds_status"])

df_pair_status = df_bounds[[c for c in PAIR_STATUS_COLUMNS if c in df_bounds.columns]].copy()
df_taylor = df_bounds.loc[
    df_bounds["support_source"].notna(), [c for c in TAYLOR_COLUMNS if c in df_bounds.columns]
].reset_index(drop=True)
df_local_affine = df_bounds.loc[
    df_bounds["local_affine_status"].notna(), [c for c in LOCAL_AFFINE_COLUMNS if c in df_bounds.columns]
].reset_index(drop=True)
df_summary = pd.DataFrame(
    [{"bounds_status": s, "count": status_counts.get(s, 0)} for s in BOUNDS_STATUSES]
)

df_pair_status.to_csv(OUTPUT_DIR / "bounds_pair_status.csv", index=False)
df_taylor.to_csv(OUTPUT_DIR / "taylor_reference_results.csv", index=False)
df_local_affine.to_csv(OUTPUT_DIR / "local_affine_results.csv", index=False)
df_summary.to_csv(OUTPUT_DIR / "bounds_summary.csv", index=False)

print(f"Wrote bounds_pair_status.csv ({len(df_pair_status)} rows)")
print(f"Wrote taylor_reference_results.csv ({len(df_taylor)} rows)")
print(f"Wrote local_affine_results.csv ({len(df_local_affine)} rows)")
print(f"Wrote bounds_summary.csv")
print()
print("Status summary:")
try:
    display(df_summary)
except NameError:
    print(df_summary.to_string(index=False))
if not df_taylor.empty:
    print("\nTaylor reference diagnostics (first 10):")
    try:
        display(df_taylor.head(10))
    except NameError:
        print(df_taylor.head(10).to_string())

## Interpreting results

### The fundamental distinction: theorem guarantee vs. sample diagnostic

**`bounds_supported` rows** use a pre-verified support interval
$[y_-, y_+]$ that contains the benchmark output almost surely.  For
analytically exact entries (e.g., Ishigami, Friedman), this is a *theorem
guarantee*: the sufficient $\eta$ bound is computed with a provably correct
support, and the projection bound is a rigorous upper bound on the Sobol
index displacement.

**`bounds_diagnostic_sample_support` rows** use the observed sample range.
The sample extremes may underestimate the true support, so the sufficient
$\eta$ bound is not guaranteed to be a valid upper bound — it is a diagnostic
estimate.  Do not report these as theorem guarantees.

### What the projection bound numbers mean

The projection bound columns give an upper bound on
$|\mathrm{PE}_{\mathscr{C}}(Z) - \mathrm{PE}_{\mathscr{C}}(V_m)|$, **not**
on $|S_u^Z - S_u^Y|$ directly (except in the $m=1$, nonzero-slope case where
$V_1$ has the same Sobol structure as $Y$).

- **When `projection_bound_s1_max` is small** (e.g., $< 0.05$): the
  transformed output's Sobol indices are close to the Taylor reference's
  indices.  For $m=1$, this also implies closeness to $Y$'s indices.
- **When the bound exceeds 1.0**: the bound is trivially satisfied but
  uninformative.  The actual shift may be much smaller.
- **When `eta_empirical_lt_one = False`**: the bound is not finite; the
  transform's nonlinearity over the output range is too large for the
  current Taylor order.

### What $\lambda$ tells you

$\lambda < 2$ is a *sufficient* condition for the local-affine bound to be
finite, not a necessary one.  A pair with $\lambda > 2$ may still have a
finite empirical $\eta_1 < 1$ and a useful bound.  The columns
`eta_empirical_lt_one` and `projection_bound_s1_max` give the bottom-line
diagnostic regardless of $\lambda$.

Practically: $\lambda \ll 1$ means the transform behaves nearly affinely over
the output's typical range, and Sobol index changes will be small.
$\lambda \approx 1$ means moderate curvature is present.  $\lambda > 2$ means
the curvature is large enough that the analytic guarantee breaks down; the
empirical $\eta$ may still give useful information.

### The reference-shift columns

`reference_shift_*` columns measure the **empirical** Sobol shift from $V_m$
to $Z$, not the theoretical bound.  These are computed from the same Saltelli
sample as the Sobol estimates.  They provide a direct, sample-based view of
how much the transformation changed the profile relative to the Taylor
reference — complementing the bound columns.

### Sampling noise

As with the noncommutativity notebook, small values of `eta_empirical`,
`projection_bound_*`, or `reference_shift_*` at `N_BASE=128` carry sampling
uncertainty.  Use `N_BASE ≥ 512` for reliable diagnostics.